# মাইক্রোসফট এজেন্ট ফ্রেমওয়ার্ক দিয়ে হিউম্যান-ইন-দ্য-লুপ ওয়ার্কফ্লো

## 🎯 শেখার উদ্দেশ্য

এই নোটবুকে, আপনি শিখবেন কীভাবে মাইক্রোসফট এজেন্ট ফ্রেমওয়ার্কের `RequestInfoExecutor` ব্যবহার করে **হিউম্যান-ইন-দ্য-লুপ** ওয়ার্কফ্লো বাস্তবায়ন করতে হয়। এই শক্তিশালী প্যাটার্ন আপনাকে AI ওয়ার্কফ্লো থামিয়ে মানুষের ইনপুট সংগ্রহ করার সুযোগ দেয়, যা আপনার এজেন্টগুলোকে ইন্টারেক্টিভ করে তোলে এবং মানুষের কাছে গুরুত্বপূর্ণ সিদ্ধান্তগুলির নিয়ন্ত্রণ দেয়।

## 🔄 হিউম্যান-ইন-দ্য-লুপ কী?

**হিউম্যান-ইন-দ্য-লুপ (HITL)** একটি ডিজাইন প্যাটার্ন যেখানে AI এজেন্টগুলি চলমান প্রক্রিয়া থামিয়ে মানুষের ইনপুট চায় তারপর চালিয়ে যায়। এটি জরুরি কারণ:

- ✅ **গুরুত্বপূর্ণ সিদ্ধান্ত** - গুরুত্বপূর্ণ কাজ করার আগে মানুষের অনুমোদন নিন
- ✅ **মিশ্র পরিস্থিতি** - যখন AI অনিশ্চিত সেখানে মানুষ স্পষ্টতা আনেন
- ✅ **ব্যবহারকারীর পছন্দ** - ব্যবহারকারীদের মাঝে বিকল্প থেকে নির্বাচন করতে বলুন
- ✅ **অনুমতি ও নিরাপত্তা** - নিয়ন্ত্রিত কাজের জন্য মানুষ নজরদারি নিশ্চিত করুন
- ✅ **ইন্টারেক্টিভ অভিজ্ঞতা** - কথোপকথনমূলক এজেন্ট তৈরি করুন যারা ব্যবহারকারীর ইনপুটের উত্তর দেয়

## 🏗️ মাইক্রোসফট এজেন্ট ফ্রেমওয়ার্ক-এ এটি কীভাবে কাজ করে

ফ্রেমওয়ার্ক হিউম্যান-ইন-দ্য-লুপের জন্য তিনটি মূল উপাদান দেয়:

1. **`RequestInfoExecutor`** - একটি বিশেষ নির্বাহক যা ওয়ার্কফ্লো থামিয়ে একটি `RequestInfoEvent` নিঃসরণ করে
2. **`RequestInfoMessage`** - টাইপ করা অনুরোধ পে-লোডগুলোর জন্য বেস ক্লাস যা মানুষের কাছে পাঠানো হয়
3. **`RequestResponse`** - মানুষের প্রতিক্রিয়া মূল অনুরোধের সঙ্গে `request_id` দিয়ে সমন্বিত করে

**ওয়ার্কফ্লো প্যাটার্ন:**
```
Agent detects need for input
    ↓
Sends message to RequestInfoExecutor
    ↓
Workflow pauses & emits RequestInfoEvent
    ↓
Application collects human input (console, UI, etc.)
    ↓
Application sends RequestResponse via send_responses_streaming()
    ↓
Workflow resumes with human input
```

## 🏨 আমাদের উদাহরণ: ব্যবহারকারীর নিশ্চিতকরণ সহ হোটেল বুকিং

আমরা শর্তাধীন ওয়ার্কফ্লোর উপরে তৈরি করব মানুষ নিশ্চিতকরণের ব্যবস্থা যুক্ত করে **বিকল্প গন্তব্য প্রস্তাব করার পূর্বে**:

1. ব্যবহারকারী গন্তব্য চায় (যেমন "প্যারিস")
2. `availability_agent` দেখে রুম আছে কিনা
3. **যদি কোন রুম না থাকে** → `confirmation_agent` জিজ্ঞাসা করে "আপনি কি বিকল্প দেখতে চান?"
4. `RequestInfoExecutor` ব্যবহার করে ওয়ার্কফ্লো **থামে**
5. **মানুষ উত্তর দেয়** "হ্যাঁ" অথবা "না" কনসোল ইনপুট দিয়ে
6. `decision_manager` প্রতিক্রিয়ার ভিত্তিতে রুট নির্ধারণ করে:
   - **হ্যাঁ** → বিকল্প গন্তব্য দেখায়
   - **না** → বুকিং অনুরোধ বাতিল করে
7. চূড়ান্ত ফলাফল প্রদর্শন করে

এটি দেখায় কিভাবে ব্যবহারকারীদের এজেন্টের প্রস্তাবের ওপর নিয়ন্ত্রণ দেওয়া যায়!

---

চলুন শুরু করা যাক! 🚀


## ধাপ ১: প্রয়োজনীয় লাইব্রেরি আমদানি

আমরা স্ট্যান্ডার্ড এজেন্ট ফ্রেমওয়ার্ক উপাদানগুলো আমদানি করি সাথে **হিউম্যান-ইন-দ্য-লুপ নির্দিষ্ট ক্লাসসমূহ**:
- `RequestInfoExecutor` - এক্সিকিউটার যা মানব ইনপুটের জন্য কার্যপ্রবাহ বিরতি দেয়
- `RequestInfoEvent` - ইভেন্ট যা মানব ইনপুট চাওয়ার সময় এমিট হয়
- `RequestInfoMessage` - টাইপ করা অনুরোধ পে-লোডের মৌলিক ক্লাস
- `RequestResponse` - মানব প্রতিক্রিয়াগুলোকে অনুরোধের সাথে মিলিয়ে দেয়
- `WorkflowOutputEvent` - কার্যপ্রবাহ আউটপুট শনাক্তকরণের জন্য ইভেন্ট


In [ ]:
import asyncio
import json
import os
from dataclasses import dataclass
from typing import Annotated, Any, Never

from agent_framework import (
    AgentExecutor,
    AgentExecutorRequest,
    AgentExecutorResponse,
    ChatMessage,
    Executor,
    RequestInfoEvent,          # NEW: Event when human input is requested
    RequestInfoExecutor,       # NEW: Executor that gathers human input
    RequestInfoMessage,        # NEW: Base class for request payloads
    RequestResponse,           # NEW: Correlates response with request
    Role,
    WorkflowBuilder,
    WorkflowContext,
    WorkflowOutputEvent,       # NEW: Event for workflow outputs
    WorkflowRunState,          # NEW: Enum of workflow run states
    WorkflowStatusEvent,       # NEW: Event for run state changes
    ai_function,
    executor,
    handler,                   # NEW: Decorator for executor methods
)

# 🤖 Azure OpenAI client integration
from agent_framework.openai import OpenAIChatClient
from dotenv import load_dotenv
from IPython.display import HTML, display
from pydantic import BaseModel

print("✅ All imports successful!")
print("🔄 Human-in-the-loop components loaded: RequestInfoExecutor, RequestInfoEvent, RequestResponse")

✅ All imports successful!
🔄 Human-in-the-loop components loaded: RequestInfoExecutor, RequestInfoEvent, RequestResponse


## ধাপ ২: কাঠামোবদ্ধ আউটপুটের জন্য পিড্যান্টিক মডেল সংজ্ঞায়িত করুন

এই মডেলগুলো সেই **স্কিমা** সংজ্ঞায়িত করে যা এজেন্টরা ফেরত দেবে। আমরা শর্তযুক্ত ওয়ার্কফ্লো থেকে সব মডেল রাখি এবং যোগ করি:

**হিউম্যান-ইন-দ্য-লুপের জন্য নতুন:**
- `HumanFeedbackRequest` - `RequestInfoMessage` এর একটি সাবক্লাস যা মানুষের কাছে পাঠানো অনুরোধ পে-লোড নির্ধারণ করে
  - এর মধ্যে থাকে `prompt` (জিজ্ঞাসা করার প্রশ্ন) এবং `destination` (অপলভ্য শহর সম্পর্কিত প্রেক্ষাপট)


In [22]:
# Existing models from conditional workflow
class BookingCheckResult(BaseModel):
    """Result from checking hotel availability at a destination."""
    destination: str
    has_availability: bool
    message: str


class AlternativeResult(BaseModel):
    """Suggested alternative destination when no rooms available."""
    alternative_destination: str
    reason: str


class BookingConfirmation(BaseModel):
    """Booking suggestion when rooms are available."""
    destination: str
    action: str
    message: str


# NEW: Pydantic model for agent's response format
class ConfirmationQuestion(BaseModel):
    """
    Pydantic model used by confirmation_agent's response_format.
    This is what the agent will output as JSON.
    """
    question: str  # The question to ask the user
    destination: str  # The unavailable destination for context


# NEW: Dataclass for RequestInfoExecutor
@dataclass
class HumanFeedbackRequest(RequestInfoMessage):
    """
    Request sent to RequestInfoExecutor asking if user wants alternatives.
    
    MUST be a dataclass subclassing RequestInfoMessage for type compatibility.
    This is what gets sent to the RequestInfoExecutor.
    """
    prompt: str = ""  # The question to ask the user
    destination: str = ""  # The unavailable destination for context


print("✅ Pydantic models defined:")
print("   - BookingCheckResult (availability check)")
print("   - AlternativeResult (alternative suggestion)")
print("   - BookingConfirmation (booking confirmation)")
print("   - ConfirmationQuestion (agent response format) 🆕")
print("   - HumanFeedbackRequest (RequestInfoMessage for HITL) 🆕")

✅ Pydantic models defined:
   - BookingCheckResult (availability check)
   - AlternativeResult (alternative suggestion)
   - BookingConfirmation (booking confirmation)
   - ConfirmationQuestion (agent response format) 🆕
   - HumanFeedbackRequest (RequestInfoMessage for HITL) 🆕


## ধাপ ৩: হোটেল বুকিং টুল তৈরি করুন

শর্তাধীন কর্মপ্রবাহ থেকে একই টুল - গন্তব্যে কক্ষ উপলব্ধ কিনা তা পরীক্ষা করে।


In [23]:
@ai_function(description="Check hotel room availability for a destination city")
def hotel_booking(destination: Annotated[str, "The destination city to check for hotel rooms"]) -> str:
    """
    Simulates checking hotel room availability.
    
    Returns JSON string with availability status.
    """
    display(
        HTML(f"""
        <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
            <strong>🔍 Tool Invoked:</strong> hotel_booking("{destination}")
        </div>
    """)
    )

    # Simulate availability check
    cities_with_rooms = ["stockholm", "seattle", "tokyo", "london", "amsterdam"]
    has_rooms = destination.lower() in cities_with_rooms

    result = {"has_availability": has_rooms, "destination": destination}

    return json.dumps(result)


print("✅ hotel_booking tool created with @ai_function decorator")

✅ hotel_booking tool created with @ai_function decorator


## ধাপ ৪: রাউটিংয়ের জন্য শর্ত ফাংশন নির্ধারণ করুন

আমাদের মানুষের মধ্যস্থতাসহ কর্মপ্রবাহের জন্য **চারটি শর্ত ফাংশন** প্রয়োজন:

**শর্তাধীন কর্মপ্রবাহ থেকে:**
১. `has_availability_condition` - যখন হোটেল উপলব্ধ থাকে তখন রাউট করে
২. `no_availability_condition` - যখন হোটেল উপলব্ধ না থাকে তখন রাউট করে

**মানুষের মধ্যস্থতায় নতুন:**
৩. `user_wants_alternatives_condition` - যখন ব্যবহারকারী বিকল্পের জন্য "হ্যাঁ" বলে তখন রাউট করে
৪. `user_declines_alternatives_condition` - যখন ব্যবহারকারী বিকল্পের জন্য "না" বলে তখন রাউট করে


In [24]:
# Existing condition functions from conditional workflow
def has_availability_condition(message: Any) -> bool:
    """Condition for routing when hotels ARE available."""
    if not isinstance(message, AgentExecutorResponse):
        return True

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)
        display(
            HTML(f"""
            <div style='padding: 12px; background: #c8e6c9; border-left: 4px solid #4caf50; border-radius: 4px; margin: 10px 0;'>
                <strong>✅ Condition Check:</strong> has_availability = <strong>{result.has_availability}</strong> for {result.destination}
            </div>
        """)
        )
        return result.has_availability
    except Exception as e:
        display(HTML(f"""<div style='padding: 12px; background: #ffcdd2; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'><strong>⚠️  Error:</strong> {str(e)}</div>"""))
        return False


def no_availability_condition(message: Any) -> bool:
    """Condition for routing when hotels are NOT available."""
    if not isinstance(message, AgentExecutorResponse):
        return False

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)
        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffecb3; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
                <strong>❌ Condition Check:</strong> no_availability for {result.destination}
            </div>
        """)
        )
        return not result.has_availability
    except Exception as e:
        return False


# NEW: Condition functions for human-in-the-loop routing
def user_wants_alternatives_condition(message: Any) -> bool:
    """
    Condition for routing when user WANTS to see alternatives.
    
    Checks the AgentExecutorRequest sent by decision_manager.
    """
    # Check if it's an AgentExecutorRequest (what decision_manager sends)
    if isinstance(message, AgentExecutorRequest):
        # Check the message text to determine user's choice
        if message.messages and len(message.messages) > 0:
            msg_text = message.messages[0].text.lower()
            wants_alternatives = "wants to see alternative" in msg_text or "want to see alternative" in msg_text
            
            display(
                HTML(f"""
                <div style='padding: 12px; background: #e1f5fe; border-left: 4px solid #0288d1; border-radius: 4px; margin: 10px 0;'>
                    <strong>🔍 User Decision:</strong> User wants alternatives = <strong>{wants_alternatives}</strong>
                </div>
            """)
            )
            
            return wants_alternatives
    
    return False
def user_declines_alternatives_condition(message: Any) -> bool:
    """
    Condition for routing when user DECLINES alternatives.
    
    Checks the AgentExecutorRequest sent by decision_manager.
    """
    # Check if it's an AgentExecutorRequest (what decision_manager sends)
    if isinstance(message, AgentExecutorRequest):
        # Check the message text to determine user's choice
        if message.messages and len(message.messages) > 0:
            msg_text = message.messages[0].text.lower()
            declined = "declined" in msg_text or "has declined" in msg_text
            
            display(
                HTML(f"""
                <div style='padding: 12px; background: #fce4ec; border-left: 4px solid #c2185b; border-radius: 4px; margin: 10px 0;'>
                    <strong>🚫 User Decision:</strong> User declined alternatives = <strong>{declined}</strong>
                </div>
            """)
            )
            
            return declined
    
    return False
print("✅ Condition functions defined:")
print("   - has_availability_condition (routes when rooms exist)")
print("   - no_availability_condition (routes when no rooms)")
print("   - user_wants_alternatives_condition (routes when user says yes) 🆕")
print("   - user_declines_alternatives_condition (routes when user says no) 🆕")

✅ Condition functions defined:
   - has_availability_condition (routes when rooms exist)
   - no_availability_condition (routes when no rooms)
   - user_wants_alternatives_condition (routes when user says yes) 🆕
   - user_declines_alternatives_condition (routes when user says no) 🆕


## ধাপ ৫: ডিসিশন ম্যানেজার এক্সিকিউটার তৈরি করুন

এটি **হিউম্যান-ইন-দ্য-লুপ প্যাটার্নের মূল অংশ**! `DecisionManager` একটি কাস্টম `Executor` যা:

১. **মিটিং অনুভূতি গ্রহন করে** `RequestResponse` অবজেক্টের মাধ্যমে
২. **ব্যবহারকারীর সিদ্ধান্ত প্রক্রিয়া করে** (হ্যাঁ/না)
৩. **ওয়ার্কফ্লো রাউট করে** উপযুক্ত এজেন্টদের কাছে মেসেজ পাঠিয়ে

মূল বৈশিষ্ট্যাবলী:
- ওয়ার্কফ্লো ধাপ হিসেবে মেথড প্রকাশের জন্য `@handler` ডেকোরেটর ব্যবহার করে
- `RequestResponse[HumanFeedbackRequest, str]` গ্রহণ করে যার মধ্যে আছে মূল অনুরোধ এবং ব্যবহারকারীর উত্তর
- সহজ "হ্যাঁ" বা "না" মেসেজ প্রদান করে যা আমাদের শর্ত ফাংশনগুলো ট্রিগার করে


In [25]:
class DecisionManager(Executor):
    """
    Coordinates workflow routing based on human feedback.
    
    This executor receives RequestResponse objects from the RequestInfoExecutor
    and makes routing decisions by sending simple messages that trigger
    condition functions.
    """

    def __init__(self, id: str | None = None):
        super().__init__(id=id or "decision_manager")

    @handler
    async def on_human_feedback(
        self,
        feedback: RequestResponse[HumanFeedbackRequest, str],
        ctx: WorkflowContext[AgentExecutorRequest],
    ) -> None:
        """
        Process human feedback and let the workflow route based on conditions.
        
        The RequestResponse contains:
        - feedback.data: The user's string reply (e.g., "yes" or "no")
        - feedback.original_request: The HumanFeedbackRequest with context
        
        This handler just displays feedback and passes the RequestResponse through.
        The routing is done by condition functions on the edges.
        """
        user_reply = (feedback.data or "").strip().lower()
        destination = getattr(feedback.original_request, "destination", "unknown")

        display(
            HTML(f"""
            <div style='padding: 15px; background: #f3e5f5; border-left: 4px solid #9c27b0; border-radius: 4px; margin: 10px 0;'>
                <strong>🎯 Decision Manager:</strong> Processing user reply: <strong>"{user_reply}"</strong> for {destination}
            </div>
        """)
        )

        if user_reply == "yes":
            display(
                HTML("""
                <div style='padding: 12px; background: #c8e6c9; border-left: 4px solid #4caf50; border-radius: 4px; margin: 10px 0;'>
                    <strong>➡️  Routing:</strong> User wants alternatives → Will route to alternative_agent
                </div>
            """)
            )
            # Create and send a message for the alternative_agent
            user_msg = ChatMessage(
                Role.USER,
                text=f"The user wants to see alternative destinations near {destination}. Please suggest one.",
            )
            await ctx.send_message(AgentExecutorRequest(messages=[user_msg], should_respond=True))
        
        elif user_reply == "no":
            display(
                HTML("""
                <div style='padding: 12px; background: #ffcdd2; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'>
                    <strong>🚫 Routing:</strong> User declined alternatives → Will route to cancellation_agent
                </div>
            """)
            )
            # Create and send a message for the cancellation_agent
            user_msg = ChatMessage(
                Role.USER,
                text="The user has declined to see alternatives. Please acknowledge their decision.",
            )
            await ctx.send_message(AgentExecutorRequest(messages=[user_msg], should_respond=True))
        
        else:
            # Handle unexpected input - treat as decline
            display(
                HTML(f"""
                <div style='padding: 12px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
                    <strong>⚠️  Warning:</strong> Unexpected input "{user_reply}" - treating as decline
                </div>
            """)
            )
            user_msg = ChatMessage(
                Role.USER,
                text="The user has declined to see alternatives. Please acknowledge their decision.",
            )
            await ctx.send_message(AgentExecutorRequest(messages=[user_msg], should_respond=True))


print("✅ DecisionManager executor created with @handler method for human feedback")

✅ DecisionManager executor created with @handler method for human feedback


## ধাপ ৬: কাস্টম ডিসপ্লে এক্সিকিউটর তৈরি করুন

শর্তসাপেক্ষ ওয়ার্কফ্লো থেকে একই ডিসপ্লে এক্সিকিউটর - ওয়ার্কফ্লো আউটপুট হিসাবে চূড়ান্ত ফলাফল দেয়।


In [26]:
@executor(id="prepare_human_request")
async def prepare_human_request(
    response: AgentExecutorResponse, 
    ctx: WorkflowContext[HumanFeedbackRequest]
) -> None:
    """
    Transform agent response into HumanFeedbackRequest for RequestInfoExecutor.
    
    This executor bridges the type gap between:
    - confirmation_agent outputs AgentExecutorResponse with ConfirmationQuestion JSON
    - request_info_executor expects HumanFeedbackRequest (RequestInfoMessage dataclass)
    """
    display(
        HTML("""
        <div style='padding: 12px; background: #e1f5fe; border-left: 4px solid #0288d1; border-radius: 4px; margin: 10px 0;'>
            <strong>🔄 Transform:</strong> Converting ConfirmationQuestion to HumanFeedbackRequest
        </div>
    """)
    )
    
    # Parse the agent's Pydantic output (ConfirmationQuestion)
    confirmation = ConfirmationQuestion.model_validate_json(response.agent_run_response.text)
    
    # Convert to HumanFeedbackRequest dataclass for RequestInfoExecutor
    feedback_request = HumanFeedbackRequest(
        prompt=confirmation.question,
        destination=confirmation.destination
    )
    
    # Send the properly typed RequestInfoMessage to the RequestInfoExecutor
    await ctx.send_message(feedback_request)


@executor(id="display_result")
async def display_result(response: AgentExecutorResponse, ctx: WorkflowContext[Never, str]) -> None:
    """
    Display the final result as workflow output.
    
    This executor receives the final agent response and yields it as the workflow output.
    """
    display(
        HTML("""
        <div style='padding: 15px; background: #f3e5f5; border-left: 4px solid #9c27b0; border-radius: 4px; margin: 10px 0;'>
            <strong>📤 Display Executor:</strong> Yielding workflow output
        </div>
    """)
    )

    await ctx.yield_output(response.agent_run_response.text)


print("✅ prepare_human_request executor created with @executor decorator")
print("✅ display_result executor created with @executor decorator")

✅ prepare_human_request executor created with @executor decorator
✅ display_result executor created with @executor decorator


## ধাপ ৭: পরিবেশ ভেরিয়েবল লোড করুন

LLM ক্লায়েন্ট (Microsoft Foundry / Azure OpenAI) কনফিগার করুন।


In [ ]:
# Load environment variables
load_dotenv()

from azure.identity import AzureCliCredential

# Azure OpenAI via the Responses API. Sign in with `az login` for keyless Entra ID auth.
# GitHub Models is deprecated (retiring July 2026) and does not support the Responses API,
# so this sample calls Azure OpenAI directly. OpenAIChatClient uses the Responses API.
chat_client = OpenAIChatClient(
    azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
    credential=AzureCliCredential(),
    model_id=os.environ["AZURE_OPENAI_DEPLOYMENT"],
)

print("✅ Chat client configured with Azure OpenAI (Responses API)")


✅ Chat client configured with GitHub Models


## ধাপ ৮: AI এজেন্ট এবং এক্সিকিউটর তৈরি করুন

আমরা **ছয়টি ওয়ার্কফ্লো উপাদান** তৈরি করি:

**এজেন্টস (AgentExecutor-এ মোড়ানো):**
১. **availability_agent** - টুল ব্যবহার করে হোটেল উপলব্ধতা পরীক্ষা করে
২. **confirmation_agent** - 🆕 মানব নিশ্চিতকরণ অনুরোধ প্রস্তুত করে
৩. **alternative_agent** - বিকল্প শহর সাজেস্ট করে (যখন ব্যবহারকারী হ্যাঁ বলেন)
৪. **booking_agent** - বুকিং উৎসাহিত করে (যখন রুম পাওয়া যায়)
৫. **cancellation_agent** - 🆕 বাতিলকরণের বার্তা পরিচালনা করে (যখন ব্যবহারকারী না বলেন)

**বিশেষ এক্সিকিউটর:**
৬. **request_info_executor** - 🆕 `RequestInfoExecutor` যা মানব ইনপুটের জন্য ওয়ার্কফ্লো থামায়
৭. **decision_manager** - 🆕 কাস্টম এক্সিকিউটর যা মানব প্রতিক্রিয়ার উপর ভিত্তি করে রুট করে (উপরেই সংজ্ঞায়িত)


In [ ]:
# Agent 1: Check availability with tool (same as conditional workflow)
availability_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a hotel booking assistant that checks room availability. "
            "Use the hotel_booking tool to check if rooms are available at the destination. "
            "Return JSON with fields: destination (string), has_availability (bool), and message (string). "
            "The message should summarize the availability status."
        ),
        tools=[hotel_booking],
        response_format=BookingCheckResult,
    ),
    id="availability_agent",
)

# Agent 2: NEW - Prepare human confirmation request
confirmation_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a helpful assistant. The user's requested destination has no available hotel rooms. "
            "Create a polite message asking if they would like to see alternative destinations nearby. "
            "Return a JSON with: destination (the unavailable city), and question (a friendly yes/no question). "
            "Keep the question concise and friendly."
        ),
        response_format=ConfirmationQuestion,  # Use Pydantic model for agent output
    ),
    id="confirmation_agent",
)

# Agent 3: Suggest alternative (when user says yes)
alternative_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a helpful travel assistant. When a user cannot find hotels in their requested city, "
            "suggest an alternative nearby city that has availability. "
            "Return JSON with fields: alternative_destination (string) and reason (string). "
            "Make your suggestion sound appealing and helpful."
        ),
        response_format=AlternativeResult,
    ),
    id="alternative_agent",
)

# Agent 4: Suggest booking (when rooms available)
booking_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a booking assistant. The user has found available hotel rooms. "
            "Encourage them to book by highlighting the destination's appeal. "
            "Return JSON with fields: destination (string), action (string), and message (string). "
            "The action should be 'book_now' and message should be encouraging."
        ),
        response_format=BookingConfirmation,
    ),
    id="booking_agent",
)

# Agent 5: NEW - Handle cancellation when user declines alternatives
class CancellationMessage(BaseModel):
    """Message when user declines alternatives."""
    status: str
    message: str

cancellation_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a helpful assistant. The user has declined to see alternative hotel destinations. "
            "Create a polite cancellation message. "
            "Return JSON with: status (should be 'cancelled'), and message (a friendly acknowledgment). "
            "Keep the message brief and understanding."
        ),
        response_format=CancellationMessage,
    ),
    id="cancellation_agent",
)

# NEW: RequestInfoExecutor - pauses workflow to gather human input
request_info_executor = RequestInfoExecutor(id="request_info")

# NEW: DecisionManager instance - routes based on human feedback
decision_manager = DecisionManager(id="decision_manager")

display(
    HTML("""
    <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
        <strong>✅ Created Workflow Components:</strong>
        <ul style='margin: 10px 0 0 0;'>
            <li><strong>availability_agent</strong> - Checks availability with hotel_booking tool</li>
            <li><strong>confirmation_agent</strong> 🆕 - Prepares human confirmation request</li>
            <li><strong>alternative_agent</strong> - Suggests alternative cities</li>
            <li><strong>booking_agent</strong> - Encourages booking</li>
            <li><strong>cancellation_agent</strong> 🆕 - Handles user declining alternatives</li>
            <li><strong>request_info_executor</strong> 🆕 - Pauses workflow for human input</li>
            <li><strong>decision_manager</strong> 🆕 - Routes based on human response</li>
        </ul>
    </div>
""")
)


## ধাপ ৯: মানব-ইন-দা-লুপ সহ ওয়ার্কফ্লো তৈরি করুন

এখন আমরা **শর্তসাপেক্ষ রাউটিং** সহ মানব-ইন-দা-লুপ পথ অন্তর্ভুক্ত করে ওয়ার্কফ্লো গ্রাফ তৈরি করছি:

**ওয়ার্কফ্লো কাঠামো:**
```
availability_agent (START)
        ↓
   Evaluate conditions
        ↙                    ↘
[no_availability]        [has_availability]
        ↓                        ↓
confirmation_agent          booking_agent
        ↓                        ↓
prepare_human_request      display_result
        ↓
request_info_executor (PAUSE)
        ↓
decision_manager
   ↙         ↘
[yes]        [no]
   ↓           ↓
alternative  cancellation
   ↓           ↓
display_result
```

**মূল এজেস:**
- `availability_agent → confirmation_agent` (কোন ঘর না থাকলে)
- `confirmation_agent → prepare_human_request` (টাইপ রূপান্তর)
- `prepare_human_request → request_info_executor` (মানবের জন্য বিরতি)
- `request_info_executor → decision_manager` (সবসময় - RequestResponse প্রদান করে)
- `decision_manager → alternative_agent` (যখন ব্যবহারকারী "হ্যাঁ" বলে)
- `decision_manager → cancellation_agent` (যখন ব্যবহারকারী "না" বলে)
- `availability_agent → booking_agent` (যখন ঘর উপলব্ধ থাকে)
- সব পথ `display_result` এ শেষ হয়


In [29]:
# Build the workflow with human-in-the-loop routing
workflow = (
    WorkflowBuilder()
    .set_start_executor(availability_agent)
    
    # NO AVAILABILITY PATH (with human-in-the-loop)
    .add_edge(availability_agent, confirmation_agent, condition=no_availability_condition)
    .add_edge(confirmation_agent, prepare_human_request)  # Transform to HumanFeedbackRequest
    .add_edge(prepare_human_request, request_info_executor)  # Send to RequestInfoExecutor
    .add_edge(request_info_executor, decision_manager)    # Always goes to decision manager
    
    # Decision manager routes based on user response
    .add_edge(decision_manager, alternative_agent, condition=user_wants_alternatives_condition)
    .add_edge(decision_manager, cancellation_agent, condition=user_declines_alternatives_condition)
    .add_edge(alternative_agent, display_result)
    .add_edge(cancellation_agent, display_result)
    
    # HAS AVAILABILITY PATH (no human input needed)
    .add_edge(availability_agent, booking_agent, condition=has_availability_condition)
    .add_edge(booking_agent, display_result)
    
    .build()
)

display(
    HTML("""
    <div style='padding: 20px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; border-radius: 8px; margin: 10px 0;'>
        <h3 style='margin: 0 0 15px 0;'>✅ Workflow Built Successfully!</h3>
        <p style='margin: 0; line-height: 1.6;'>
            <strong>Human-in-the-Loop Routing:</strong><br>
            • If <strong>NO availability</strong> → confirmation_agent → prepare_human_request → request_info_executor → <strong>PAUSE FOR HUMAN</strong> → decision_manager<br>
            &nbsp;&nbsp;• If user says <strong>YES</strong> → alternative_agent → display_result<br>
            &nbsp;&nbsp;• If user says <strong>NO</strong> → cancellation_agent → display_result<br>
            • If <strong>availability</strong> → booking_agent → display_result (no human input needed)
        </p>
    </div>
""")
)

## ধাপ ১০: টেস্ট কেস ১ চালান - সিটি উইথআউট অ্যাভেইলেবিলিটি (প্যারিস হিউম্যান কনফার্মেশনের সাথে)

এই পরীক্ষা প্রদর্শন করে **সম্পূর্ণ মানব-ইন-দ্য-লুপ চক্র**:

১. প্যারিসে হোটেল অনুরোধ
২. availability_agent চেক করে → কোনো রুম নেই
৩. confirmation_agent মানব-মুখোমুখি প্রশ্ন তৈরি করে
৪. request_info_executor **ওয়ার্কফ্লো থামে** এবং `RequestInfoEvent` নির্গত করে
৫. **অ্যাপ্লিকেশন ইভেন্টটি শনাক্ত করে এবং কনসোলে ব্যবহারকারীকে প্রম্পট করে**
৬. ব্যবহারকারী "yes" বা "no" টাইপ করে
৭. অ্যাপ্লিকেশন `send_responses_streaming()` এর মাধ্যমে প্রতিক্রিয়া পাঠায়
৮. decision_manager প্রতিক্রিয়ার ভিত্তিতে রুটিং করে
৯. চূড়ান্ত ফলাফল প্রদর্শিত হয়

**মূল প্যাটার্ন:**
- প্রথম পুনরাবৃত্তির জন্য `workflow.run_stream()` ব্যবহার করুন
- পরবর্তী পুনরাবৃত্তির জন্য `workflow.send_responses_streaming(pending_responses)` ব্যবহার করুন
- মানব ইনপুটের প্রয়োজন হলে তা সনাক্ত করতে `RequestInfoEvent` শোনুন
- চূড়ান্ত ফলাফল ধরতে `WorkflowOutputEvent` শোনুন


In [ ]:
display(
    HTML("""
    <div style='padding: 20px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #e65100;'>🧪 TEST CASE 1: Paris (No Availability - Human-in-the-Loop)</h3>
        <p style='margin: 0;'>Expected workflow path: availability_agent → confirmation_agent → request_info_executor → <strong>PAUSE</strong> → decision_manager → (depends on user input)</p>
    </div>
""")
)

# Create request for Paris
request_paris = AgentExecutorRequest(
    messages=[ChatMessage(Role.USER, text="I want to book a hotel in Paris")], 
    should_respond=True
)

# Human-in-the-loop execution pattern
pending_responses: dict[str, str] | None = None
completed = False
workflow_output: str | None = None

print("\n🔄 Starting human-in-the-loop workflow...")
print("=" * 60)

while not completed:
    # First iteration uses run_stream with the request
    # Subsequent iterations use send_responses_streaming with collected human responses
    if pending_responses:
        print(f"\n📤 Sending human responses: {pending_responses}")
        stream = workflow.send_responses_streaming(pending_responses)
        pending_responses = None  # Clear immediately after sending
    else:
        print(f"\n🚀 Starting workflow with request: 'I want to book a hotel in Paris'")
        stream = workflow.run_stream(request_paris)
    
    # Collect all events from this iteration
    events = [event async for event in stream]
    
    # Process events
    requests: list[tuple[str, str]] = []  # (request_id, prompt)
    
    for event in events:
        # Check for human input requests
        if isinstance(event, RequestInfoEvent) and isinstance(event.data, HumanFeedbackRequest):
            print(f"\n⏸️  WORKFLOW PAUSED - Human input requested!")
            print(f"   Request ID: {event.request_id}")
            print(f"   Destination: {event.data.destination}")
            requests.append((event.request_id, event.data.prompt))
        
        # Check for workflow outputs
        elif isinstance(event, WorkflowOutputEvent):
            workflow_output = str(event.data)
            completed = True
            print(f"\n✅ Workflow completed with output!")
    
    # If we have human requests, prompt the user
    if requests and not completed:
        responses: dict[str, str] = {}
        for req_id, prompt in requests:
            print(f"\n{'='*60}")
            print(f"💬 QUESTION FOR YOU:")
            print(f"   {prompt}")
            print(f"{'='*60}")
            
            # Get user input (in notebook, this will pause execution)
            answer = input("👉 Enter 'yes' or 'no': ").strip().lower()
            
            print(f"\n📝 You answered: {answer}")
            responses[req_id] = answer
        
        pending_responses = responses

print(f"\n{'='*60}")
print(f"🏆 FINAL WORKFLOW OUTPUT:")
print(f"{'='*60}")

# Display final result
if workflow_output:
    # Try to parse as JSON for pretty display
    try:
        result_data = json.loads(workflow_output)
        if "alternative_destination" in result_data:
            result_obj = AlternativeResult.model_validate_json(workflow_output)
            display(
                HTML(f"""
                <div style='padding: 25px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); border-radius: 12px; box-shadow: 0 4px 12px rgba(255,165,0,0.3); margin: 20px 0;'>
                    <h3 style='margin: 0 0 15px 0; color: #333;'>🏆 WORKFLOW RESULT</h3>
                    <div style='background: white; padding: 20px; border-radius: 8px;'>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ❌ No rooms in Paris</p>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>User Decision:</strong> ✅ Accepted alternatives</p>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Alternative Suggestion:</strong> 🏨 {result_obj.alternative_destination}</p>
                        <p style='margin: 0; font-size: 14px; color: #666;'><strong>Reason:</strong> {result_obj.reason}</p>
                    </div>
                </div>
            """)
            )
        else:
            # User declined
            display(
                HTML(f"""
                <div style='padding: 25px; background: linear-gradient(135deg, #f44336 0%, #e91e63 100%); color: white; border-radius: 12px; box-shadow: 0 4px 12px rgba(244,67,54,0.3); margin: 20px 0;'>
                    <h3 style='margin: 0 0 15px 0;'>🏆 WORKFLOW RESULT</h3>
                    <div style='background: white; color: #333; padding: 20px; border-radius: 8px;'>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ❌ No rooms in Paris</p>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>User Decision:</strong> 🚫 Declined alternatives</p>
                        <p style='margin: 0; font-size: 14px; color: #666;'><strong>Result:</strong> Booking request cancelled</p>
                    </div>
                </div>
            """)
            )
    except:
        print(workflow_output)


🔄 Starting human-in-the-loop workflow...

🚀 Starting workflow with request: 'I want to book a hotel in Paris'



⏸️  WORKFLOW PAUSED - Human input requested!
   Request ID: 032c8fce-b9d1-400e-ba8d-afd2248e2926
   Destination: Paris

💬 QUESTION FOR YOU:
   Unfortunately, there are no rooms available in Paris. Would you like to explore nearby alternative destinations?

📝 You answered: yes

📤 Sending human responses: {'032c8fce-b9d1-400e-ba8d-afd2248e2926': 'yes'}

🚀 Starting workflow with request: 'I want to book a hotel in Paris'

📝 You answered: yes

📤 Sending human responses: {'032c8fce-b9d1-400e-ba8d-afd2248e2926': 'yes'}

🚀 Starting workflow with request: 'I want to book a hotel in Paris'



⏸️  WORKFLOW PAUSED - Human input requested!
   Request ID: cf48dad0-ee5e-4f60-8806-341a7a292bd4
   Destination: Paris

💬 QUESTION FOR YOU:
   I'm sorry to inform you that there are no available hotel rooms in Paris. Would you like me to suggest nearby alternative destinations?

📝 You answered: 

📤 Sending human responses: {'cf48dad0-ee5e-4f60-8806-341a7a292bd4': ''}

🚀 Starting workflow with request: 'I want to book a hotel in Paris'

📝 You answered: 

📤 Sending human responses: {'cf48dad0-ee5e-4f60-8806-341a7a292bd4': ''}

🚀 Starting workflow with request: 'I want to book a hotel in Paris'


## ধাপ ১১: টেস্ট কেস ২ চালান - সিটি WITH অ্যাভাইলেবিলিটি (স্টকহোম - কোন মানব ইনপুট প্রয়োজন নেই)

এই টেস্টটি **সরাসরি পথ** প্রদর্শন করে যখন রুমগুলি উপলব্ধ থাকে:

১. স্টকহোমে হোটেল অনুরোধ করুন
২. availability_agent পরীক্ষা করে → রুমগুলি উপলব্ধ ✅
৩. booking_agent বুকিং প্রস্তাব করে
৪. display_result নিশ্চিতকরণ প্রদর্শন করে
৫. **কোনো মানব ইনপুট প্রয়োজন নেই!**

যখন রুমগুলি উপলব্ধ থাকে, তখন ওয়ার্কফ্লো সম্পূর্ণরূপে মানব-ইন-দ্য-লুপ পথ বাইপাস করে।


In [ ]:
display(
    HTML("""
    <div style='padding: 20px; background: #e8f5e9; border-left: 4px solid #4caf50; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #1b5e20;'>🧪 TEST CASE 2: Stockholm (Has Availability - No Human Input)</h3>
        <p style='margin: 0;'>Expected workflow path: availability_agent → booking_agent → display_result (direct, no pause)</p>
    </div>
""")
)

# Create request for Stockholm
request_stockholm = AgentExecutorRequest(
    messages=[ChatMessage(Role.USER, text="I want to book a hotel in Stockholm")], 
    should_respond=True
)

# Run the workflow (should complete without human input)
events_stockholm = await workflow.run(request_stockholm)
outputs_stockholm = events_stockholm.get_outputs()

# Display results
if outputs_stockholm:
    result_stockholm = BookingConfirmation.model_validate_json(outputs_stockholm[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #4caf50 0%, #8bc34a 100%); color: white; border-radius: 12px; box-shadow: 0 4px 12px rgba(76,175,80,0.3); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0;'>🏆 WORKFLOW RESULT (Stockholm - No Human Input)</h3>
            <div style='background: white; color: #333; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ✅ Rooms Available!</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Destination:</strong> 🏨 {result_stockholm.destination}</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Action:</strong> {result_stockholm.action}</p>
                <p style='margin: 0 0 10px 0; font-size: 14px; color: #666;'><strong>Message:</strong> {result_stockholm.message}</p>
                <p style='margin: 10px 0 0 0; font-size: 12px; color: #999; font-style: italic;'>Note: No human input was requested because rooms were available!</p>
            </div>
        </div>
    """)
    )

## মূল শিখনযোগ্য বিষয় এবং মানব-ইন-দ্য-লুপ সেরা অনুশীলনসমূহ

### ✅ আপনি যা শিখেছেন:

#### 1. **RequestInfoExecutor প্যাটার্ন**
Microsoft Agent Framework-এ মানব-ইন-দ্য-লুপ প্যাটার্ন তিনটি মূল উপাদান ব্যবহার করে:
- `RequestInfoExecutor` - ওয়ার্কফ্লো থামায় এবং ইভেন্ট সৃষ্টি করে
- `RequestInfoMessage` - টাইপকৃত রিকোয়েস্ট পেলোডের বেস ক্লাস (এটিকে সাবক্লাস করুন!)
- `RequestResponse` - মানবের প্রতিক্রিয়াগুলো মূল অনুরোধের সাথে সম্পর্কিত করে

**গুরুত্বপূর্ণ বোধগম্যতা:**
- `RequestInfoExecutor` নিজেই ইনপুট সংগ্রহ করে না - এটি শুধুমাত্র ওয়ার্কফ্লো থামায়
- আপনার অ্যাপ্লিকেশন কোডকে `RequestInfoEvent` শোনার এবং ইনপুট সংগ্রহ করার জন্য থাকতে হবে
- আপনাকে `send_responses_streaming()` কল করতে হবে একটি dict প্রদান করে যা `request_id` থেকে ব্যবহারকারীর উত্তর ম্যাপ করে

#### 2. **স্ট্রিমিং এক্সিকিউশন প্যাটার্ন**
```python
# প্রথম পুনরাবৃত্তি
stream = workflow.run_stream(initial_request)

# পরবর্তী পুনরাবৃত্তি (মানব ইনপুটের পরে)
stream = workflow.send_responses_streaming(pending_responses)

# সবসময় ইভেন্টগুলি প্রক্রিয়াজাত করুন
events = [event async for event in stream]
```

#### 3. **ইভেন্ট-চালিত আর্কিটেকচার**
নির্দিষ্ট ইভেন্টগুলোর জন্য শোনুন ওয়ার্কফ্লো নিয়ন্ত্রণের জন্য:
- `RequestInfoEvent` - মানব ইনপুট দরকার (ওয়ার্কফ্লো থামানো হয়েছে)
- `WorkflowOutputEvent` - চূড়ান্ত ফলাফল উপলব্ধ (ওয়ার্কফ্লো সম্পন্ন)
- `WorkflowStatusEvent` - অবস্থা পরিবর্তনসমূহ (IN_PROGRESS, IDLE_WITH_PENDING_REQUESTS, ইত্যাদি)

#### 4. **@handler সহ কাস্টম Executor সমূহ**
`DecisionManager` দেখায় কিভাবে Executor তৈরি করবেন যেগুলি:
- `@handler` ডেকোরেটর ব্যবহার করে পদ্ধতিগুলোকে ওয়ার্কফ্লো স্টেপ হিসেবে প্রকাশ করে
- টাইপকৃত মেসেজ গ্রহণ করে (যেমন, `RequestResponse[HumanFeedbackRequest, str]`)
- অন্য Executor-এ মেসেজ পাঠিয়ে ওয়ার্কফ্লো রুট করে
- `WorkflowContext` এর মাধ্যমে প্রসঙ্গ অ্যাক্সেস করে

#### 5. **মানব সিদ্ধান্ত সহ শর্তাধীন রাউটিং**
আপনি এমন শর্তমূলক ফাংশন তৈরি করতে পারেন যা মানব প্রতিক্রিয়া মূল্যায়ন করে:
```python
def user_wants_alternatives_condition(message: Any) -> bool:
    response_text = message.agent_run_response.text.lower()
    return response_text == "yes"
```

### 🎯 বাস্তব-বিশ্বের অ্যাপ্লিকেশনসমূহ:

1. **অনুমোদন ওয়ার্কফ্লোসমূহ**
   - ব্যয় প্রতিবেদন প্রক্রিয়াকরণের আগে ব্যবস্থাপক অনুমোদন নিন
   - স্বয়ংক্রিয় ইমেইল পাঠানোর আগে মানব পর্যালোচনা প্রয়োজন
   - উচ্চ-মূল্যের লেনদেন কার্যকর করার আগে নিশ্চিত করুন

2. **বিষয়বস্তু পরিদর্শন**
   - সন্দেহজনক বিষয়বস্তু মানব পর্যালোচনার জন্য চিহ্নিত করুন
   - প্রান্তিক কেসে চূড়ান্ত সিদ্ধান্ত নিতে পরিষেবাদানকারীদের কাছে জিজ্ঞাসা করুন
   - AI আত্মবিশ্বাস কম থাকলে মানবদের কাছে প্রসারিত করুন

3. **গ্রাহক সেবা**
   - AI স্বয়ংক্রিয়ভাবে রুটিন প্রশ্নগুলো পরিচালনা করুক
   - জটিল সমস্যা মানব এজেন্টদের কাছে প্রসারিত করুন
   - গ্রাহকের কাছে প্রশ্ন করুন তারা কি মানবের সাথে কথা বলতে চান

4. **তথ্য প্রক্রিয়াকরণ**
   - অস্পষ্ট তথ্য এন্ট্রি সৃষ্টির জন্য মানুষদের জিজ্ঞাসা করুন
   - অস্পষ্ট নথির AI ব্যাখ্যার নিশ্চিতকরণ করুন
   - ব্যবহারকারীদের মধ্যে বিভিন্ন বৈধ ব্যাখ্যার মধ্যে বেছে নিতে দিন

5. **নিরাপত্তা-সমালোচনামূলক সিস্টেমসমূহ**
   - অপরিবর্তনীয় ক্রিয়াকলাপের আগে মানব নিশ্চিতকরণ প্রয়োজন
   - সংবেদনশীল ডেটা অ্যাক্সেস করার আগে অনুমোদন নিন
   - নিয়ন্ত্রিত শিল্পে (স্বাস্থ্যসেবা, অর্থ) সিদ্ধান্ত নিশ্চিত করুন

6. **ইন্টারেক্টিভ এজেন্টসমূহ**
   - কথোপকথনমূলক বট তৈরি করুন যেগুলি ফলো-আপ প্রশ্ন জিজ্ঞাসা করে
   - ব্যবহারকারীদের জটিল প্রক্রিয়াগুলোর মাধ্যমে গাইড করার জন্য উইজার্ড তৈরি করুন
   - এমন এজেন্ট ডিজাইন করুন যারা ধাপে ধাপে মানুষের সাথে সহযোগিতা করে

### 🔄 তুলনা: মানব-ইন-দ্য-লুপ সহ বনাম ছাড়া

| বৈশিষ্ট্য | শর্তাধীন ওয়ার্কফ্লো | মানব-ইন-দ্য-লুপ ওয়ার্কফ্লো |
|---------|---------------------|---------------------------|
| **কার্যকরী সমাপন** | একক `workflow.run()` | লুপ with `run_stream()` + `send_responses_streaming()` |
| **ব্যবহারকারী ইনপুট** | নেই (সম্পূর্ণ স্বয়ংক্রিয়) | ইন্টারেক্টিভ প্রম্পট `input()` বা UI এর মাধ্যমে |
| **উপাদানগুলি** | এজেন্ট + Executor | + RequestInfoExecutor + DecisionManager |
| **ইভেন্টসমূহ** | AgentExecutorResponse মাত্র | RequestInfoEvent, WorkflowOutputEvent, ইত্যাদি |
| **বিরতি** | বিরতি নেই | RequestInfoExecutor এ ওয়ার্কফ্লো থামে |
| **মানব নিয়ন্ত্রণ** | মানব নিয়ন্ত্রণ নেই | মানুষ মূল সিদ্ধান্ত নেয় |
| **ব্যবহারের ক্ষেত্র** | স্বয়ংক্রিয়করণ | সহযোগিতা ও তদারকি |

### 🚀 উন্নত প্যাটার্নসমূহ:

#### একাধিক মানব সিদ্ধান্ত পয়েন্ট
একই ওয়ার্কফ্লোতে একাধিক `RequestInfoExecutor` নোড থাকতে পারে:
```python
.add_edge(agent1, request_info_1)  # প্রথম মানব সিদ্ধান্ত
.add_edge(decision_manager_1, agent2)
.add_edge(agent2, request_info_2)  # দ্বিতীয় মানব সিদ্ধান্ত
.add_edge(decision_manager_2, final_agent)
```

#### টাইমআউট হ্যান্ডলিং
মানব প্রতিক্রিয়ার জন্য টাইমআউট প্রয়োগ করুন:
```python
import asyncio

try:
    answer = await asyncio.wait_for(
        asyncio.to_thread(input, "Enter yes/no: "),
        timeout=60.0
    )
except asyncio.TimeoutError:
    answer = "no"  # ডিফল্ট নিরাপদ বিকল্প হিসেবে
```

#### সমৃদ্ধ UI ইন্টিগ্রেশন
`input()` এর পরিবর্তে ওয়েব UI, Slack, Teams ইত্যাদির সাথে ইন্টিগ্রেট করুন:
```python
if isinstance(event, RequestInfoEvent):
    # ব্যবহারকারীর পছন্দসই চ্যানেলে বিজ্ঞপ্তি পাঠান
    await slack_client.send_message(
        user_id=current_user,
        text=event.data.prompt,
        request_id=event.request_id
    )
```

#### শর্তাধীন মানব-ইন-দ্য-লুপ
নির্দিষ্ট পরিস্থিতিতে মাত্র মানব ইনপুট চাইবেন:
```python
def needs_human_approval_condition(message: Any) -> bool:
    # কেবল তখনই মানুষের কাছে রুট করুন যখন বিশ্বাসের মাত্রা কম বা মান বেশি হয়
    if result.confidence < 0.7 or result.value > 10000:
        return True
    return False
```

### ⚠️ সেরা অনুশীলনসমূহ:

1. **দয়া করে সর্বদা RequestInfoMessage সাবক্লাস করুন**
   - টাইপ নিরাপত্তা এবং যাচাই প্রদান করে
   - UI রেন্ডারিং এর জন্য সমৃদ্ধ প্রসঙ্গ সক্ষম করে
   - প্রতিটি রিকোয়েস্ট টাইপের উদ্দেশ্য স্পষ্ট করে

2. **বর্ণনামূলক প্রম্পট ব্যবহার করুন**
   - আপনি যা জিজ্ঞাসা করছেন তার প্রসঙ্গ অন্তর্ভুক্ত করুন
   - প্রতিটি বিকল্পের পরিণতি ব্যাখ্যা করুন
   - প্রশ্নগুলো সহজ এবং স্পষ্ট রাখুন

3. **অপ্রত্যাশিত ইনপুট হ্যান্ডলিং করুন**
   - ব্যবহারকারীর উত্তর যাচাই করুন
   - অবৈধ ইনপুটের জন্য ডিফল্ট দিন
   - পরিষ্কার ত্রুটি বার্তা দিন

4. **রিকোয়েস্ট আইডি ট্র্যাক করুন**
   - `request_id` এবং প্রতিক্রিয়ার মধ্যকার সম্পর্ক ব্যবহার করুন
   - ম্যানুয়ালি স্টেট পরিচালনা করার চেষ্ট করবেন না

5. **নন-ব্লকিং ডিজাইন করুন**
   - ইনপুটের জন্য থ্রেড ব্লক করবেন না
   - সার্বক্ষণিক async প্যাটার্ন ব্যবহার করুন
   - সমান্তরাল ওয়ার্কফ্লো ইন্সট্যান্স সমর্থন করুন

### 📚 সম্পর্কিত ধারণাসমূহ:

- **এজেন্ট মিডলওয়্যার** - এজেন্ট কল ইন্টারসেপ্ট করা (পূর্ববর্তী নোটবুক)
- **ওয়ার্কফ্লো স্টেট ম্যানেজমেন্ট** - রানের মধ্যে ওয়ার্কফ্লো অবস্থা সংরক্ষণ
- **মাল্টি-এজেন্ট সহযোগিতা** - মানব-ইন-দ্য-লুপ এজেন্ট টিমের সাথে সংযুক্ত করুন
- **ইভেন্ট-চালিত আর্কিটেকচারসমূহ** - ইভেন্টের মাধ্যমে প্রতিক্রিয়াশীল সিস্টেম নির্মাণ

---

### 🎓 অভিনন্দন!

আপনি Microsoft Agent Framework দিয়ে মানব-ইন-দ্য-লুপ ওয়ার্কফ্লো দক্ষভাবে আয়ত্ত করেছেন! আপনি এখন জানেন কিভাবে:
- ✅ মানব ইনপুট সংগ্রহের জন্য ওয়ার্কফ্লো থামাবেন
- ✅ RequestInfoExecutor এবং RequestInfoMessage ব্যবহার করবেন
- ✅ ইভেন্টস এর মাধ্যমে স্ট্রিমিং এক্সিকিউশন হ্যান্ডল করবেন
- ✅ @handler দিয়ে কাস্টম Executor তৈরি করবেন
- ✅ মানব সিদ্ধান্তের উপর ভিত্তি করে ওয়ার্কফ্লো রুট করবেন
- ✅ মানবদের সাথে সহযোগিতা করে ইন্টারেক্টিভ AI এজেন্ট তৈরি করবেন

**এটি বিশ্বাসযোগ্য, নিয়ন্ত্রনযোগ্য AI সিস্টেম নির্মাণের একটি গুরুত্বপূর্ণ প্যাটার্ন!** 🚀


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**অস্বীকৃতি**:
এই নথিটি AI অনুবাদ পরিষেবা [Co-op Translator](https://github.com/Azure/co-op-translator) ব্যবহার করে অনূদিত হয়েছে। যদিও আমরা শুদ্ধতার জন্য চেষ্টা করি, অনুগ্রহ করে মনে রাখবেন যে স্বয়ংক্রিয় অনুবাদে ত্রুটি বা অসঙ্গতি থাকতে পারে। মূল নথিটি তার স্বভাষায় কর্তৃত্বপূর্ণ উৎস হিসেবে বিবেচিত হওয়া উচিত। গুরুত্বপূর্ণ তথ্যের জন্য পেশাদার মানব অনুবাদ সুপারিশ করা হয়। এই অনুবাদের ব্যবহারে প্রয়োজনীয় ভুল বোঝাবুঝি বা ভুল ব্যাখ্যার জন্য আমরা দায়বদ্ধ নই।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
